# ASCII Stream Engine (Control v2)

Notebook con control de filtros, presets y modo **RAW (sin ASCII)**.

**Pasos:**
1. `python -m pip install -r ascii_stream_engine/requirements.txt`
2. Verifica `ffmpeg`
3. VLC: `udp://@127.0.0.1:1234`

**Notas:**
- Los filtros se aplican **antes** del ASCII.
- Si desactivas filtros, queda imagen normal (sin efectos).
- `render_mode="raw"` manda imagen sin ASCII.
- En el tab **Engine** podes cambiar la camara de entrada.


In [ ]:
import os
import sys

repo_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print("Repo root:", repo_root)


In [ ]:
import os

from ascii_stream_engine import (
    EngineConfig,
    StreamEngine,
    OpenCVCameraSource,
    AsciiRenderer,
    FfmpegUdpOutput,
    FilterPipeline,
    AnalyzerPipeline,
)
from ascii_stream_engine.adapters.processors import (
    BrightnessFilter,
    EdgeFilter,
    InvertFilter,
)

# DetailBoost puede no existir si no actualizaste el repo
try:
    from ascii_stream_engine.adapters.processors import DetailBoostFilter
except Exception:
    DetailBoostFilter = None

# Renderer con fuente monoespaciada (si existe)
FONT_PATH = "/usr/share/fonts/truetype/freefont/FreeMono.ttf"
renderer = AsciiRenderer()
if os.path.exists(FONT_PATH):
    try:
        renderer = AsciiRenderer(font_path=FONT_PATH)
    except TypeError:
        from PIL import ImageFont
        renderer = AsciiRenderer(font=ImageFont.truetype(FONT_PATH, 12))

config = EngineConfig(host="127.0.0.1", port=1234, fps=30, grid_w=100, grid_h=31)
filters = FilterPipeline([])
analyzers = AnalyzerPipeline([])

engine = StreamEngine(
    source=OpenCVCameraSource(0),
    renderer=renderer,
    sink=FfmpegUdpOutput(),
    config=config,
    analyzers=analyzers,
    filters=filters,
)


In [ ]:
import ipywidgets as widgets
from IPython.display import display

detail_filter = DetailBoostFilter() if DetailBoostFilter else None
edge_filter = EdgeFilter(60, 120)
brightness_filter = BrightnessFilter()
invert_filter = InvertFilter()

FILTERS = {
    "Edges": edge_filter,
    "Brightness/Contrast": brightness_filter,
    "Invert": invert_filter,
}
if detail_filter:
    FILTERS["Detail Boost"] = detail_filter

filter_checkboxes = {
    name: widgets.Checkbox(value=False, description=name) for name in FILTERS
}

status = widgets.HTML()

DENSE_CHARSET = " .'`^\\\",:;Il!i~+_-?][}{1)(|\\\\/tfjrxnuvczXYUJCLQ0OZmwqpdbkhao*#MW&8%B@$"
PRESETS = {
    "Performance": dict(fps=30, grid_w=80, grid_h=25, charset=" .:-=+*#"),
    "Balance": dict(fps=30, grid_w=100, grid_h=31, charset=" .:-=+*#%@"),
    "Alta definición": dict(fps=24, grid_w=140, grid_h=44, charset=DENSE_CHARSET),
    "Baja latencia": dict(
        fps=30,
        grid_w=80,
        grid_h=25,
        charset=" .:-=+*#",
        frame_buffer_size=0,
        bitrate="800k",
    ),
}

preset_dd = widgets.Dropdown(options=list(PRESETS.keys()), value="Balance")
apply_preset_btn = widgets.Button(description="Aplicar preset")

fps_slider = widgets.IntSlider(value=engine.get_config().fps, min=10, max=60, description="FPS")
grid_w_slider = widgets.IntSlider(value=engine.get_config().grid_w, min=60, max=200, description="Grid W")
grid_h_slider = widgets.IntSlider(value=engine.get_config().grid_h, min=20, max=120, description="Grid H")

charset_options = {
    "Simple": " .:-=+*#",
    "Medio": " .:-=+*#%@",
    "Denso": DENSE_CHARSET,
}
current_charset = engine.get_config().charset
if current_charset not in charset_options.values():
    charset_options["Actual"] = current_charset
charset_default = current_charset if current_charset in charset_options.values() else charset_options["Simple"]

charset_dd = widgets.Dropdown(
    options=charset_options,
    value=charset_default,
    description="Charset",
)

render_mode = widgets.RadioButtons(
    options=[("ASCII", "ascii"), ("RAW (sin ASCII)", "raw")],
    value="ascii",
    description="Modo",
)

raw_width = widgets.IntText(value=640, description="Raw W")
raw_height = widgets.IntText(value=360, description="Raw H")
raw_use_size = widgets.Checkbox(value=False, description="Usar tamaño RAW")

contrast_slider = widgets.FloatSlider(value=engine.get_config().contrast, min=0.5, max=3.0, step=0.1, description="Contraste")
brightness_slider = widgets.IntSlider(value=engine.get_config().brightness, min=-50, max=50, step=1, description="Brillo")
frame_buffer_slider = widgets.IntSlider(value=engine.get_config().frame_buffer_size, min=0, max=3, step=1, description="Buffer")
bitrate_text = widgets.Text(value=str(engine.get_config().bitrate), description="Bitrate")

apply_ascii_btn = widgets.Button(description="Aplicar ajustes")
clear_filters_btn = widgets.Button(description="Quitar filtros")
start_btn = widgets.Button(description="Start")
stop_btn = widgets.Button(description="Stop")

camera_index = widgets.IntText(value=0, description="Camara")
apply_camera_btn = widgets.Button(description="Aplicar camara")


def rebuild_filters(_=None) -> None:
    selected = [FILTERS[name] for name, cb in filter_checkboxes.items() if cb.value]
    engine.filter_pipeline.replace(selected)
    if selected:
        status.value = f"Filtros activos: {', '.join([f.name for f in selected])}"
    else:
        status.value = "Sin filtros: imagen normal (sin efectos)."


def apply_preset(_=None) -> None:
    preset = PRESETS[preset_dd.value]
    was_running = engine.is_running
    if was_running:
        engine.stop()
    engine.update_config(**preset)
    fps_slider.value = preset["fps"]
    grid_w_slider.value = preset["grid_w"]
    grid_h_slider.value = preset["grid_h"]
    charset_dd.value = preset["charset"]
    if "frame_buffer_size" in preset:
        frame_buffer_slider.value = preset["frame_buffer_size"]
    if "bitrate" in preset:
        bitrate_text.value = preset["bitrate"]
    if was_running:
        engine.start()
    status.value = f"Preset aplicado: {preset_dd.value}"


def apply_settings(_=None) -> None:
    was_running = engine.is_running
    if was_running:
        engine.stop()
    raw_w = raw_width.value if raw_use_size.value else None
    raw_h = raw_height.value if raw_use_size.value else None
    engine.update_config(
        fps=fps_slider.value,
        grid_w=grid_w_slider.value,
        grid_h=grid_h_slider.value,
        charset=charset_dd.value,
        contrast=contrast_slider.value,
        brightness=brightness_slider.value,
        render_mode=render_mode.value,
        raw_width=raw_w,
        raw_height=raw_h,
        frame_buffer_size=frame_buffer_slider.value,
        bitrate=bitrate_text.value,
    )
    if was_running:
        engine.start()
    status.value = "Ajustes aplicados (puede requerir reconectar VLC)."


def clear_filters(_=None) -> None:
    for cb in filter_checkboxes.values():
        cb.value = False
    rebuild_filters()


def start_engine(_=None) -> None:
    engine.start()
    status.value = "Engine corriendo."


def stop_engine(_=None) -> None:
    engine.stop()
    status.value = "Engine detenido."


def apply_camera(_=None) -> None:
    source = engine.get_source()
    if not hasattr(source, "set_camera_index"):
        status.value = "Fuente actual no soporta cambio de camara."
        return
    was_running = engine.is_running
    if was_running:
        engine.stop()
    source.set_camera_index(int(camera_index.value))
    if was_running:
        engine.start()
    status.value = f"Camara cambiada a {camera_index.value}."


apply_preset_btn.on_click(apply_preset)
apply_ascii_btn.on_click(apply_settings)
clear_filters_btn.on_click(clear_filters)
start_btn.on_click(start_engine)
stop_btn.on_click(stop_engine)
apply_camera_btn.on_click(apply_camera)

for cb in filter_checkboxes.values():
    cb.observe(rebuild_filters, names="value")

engine_box = widgets.VBox(
    [
        widgets.HBox([start_btn, stop_btn]),
        widgets.HBox([camera_index, apply_camera_btn]),
        status,
    ]
)
filters_box = widgets.VBox(
    [widgets.HTML("<b>Filtros (antes de ASCII)</b>")] +
    list(filter_checkboxes.values()) +
    [clear_filters_btn]
)

ascii_box = widgets.VBox(
    [
        widgets.HTML("<b>ASCII / Presets</b>"),
        widgets.HBox([preset_dd, apply_preset_btn]),
        fps_slider,
        grid_w_slider,
        grid_h_slider,
        charset_dd,
        contrast_slider,
        brightness_slider,
        frame_buffer_slider,
        bitrate_text,
        widgets.HTML("<b>Modo de salida</b>"),
        render_mode,
        raw_use_size,
        widgets.HBox([raw_width, raw_height]),
        apply_ascii_btn,
    ]
)

tabs = widgets.Tab(children=[engine_box, filters_box, ascii_box])
tabs.set_title(0, "Engine")
tabs.set_title(1, "Filtros")
tabs.set_title(2, "ASCII/RAW")

rebuild_filters()
display(tabs)


In [ ]:
# Usa el panel para iniciar/detener el engine.
